# Priminfo archive: scraping the premium PDFs

## What this notebook does

We download the yearly premium PDFs from the archive of the Federal Office of Public Health (BAG):
https://www.priminfo.admin.ch/de/downloads/archiv

Then we open one of them with **pdfplumber** to see how the tables inside look.

The PDFs are our backup route for the premium data. Our main source is `Praemien_CH.csv` from
opendata.swiss. The PDFs let us cross-check those numbers, and they show the scraping part of the
course project.

# Before start:
1. The notebook must be in the folder `CIP-project\notebooks\`.
2. In VS Code, click **"Select Kernel"** at the top right and choose the Python from `.venv`.
   That is where `requests`, `beautifulsoup4` and `pdfplumber` are installed.
3. `data/raw/` must be listed in `.gitignore`. Otherwise the downloaded PDFs end up in Git

In [26]:
# Imports 
import csv
import random
import time
from datetime import datetime
from pathlib import Path
from urllib.parse import urljoin, urlparse   # urljoin turns "/de/x.pdf" into a full web address
from urllib.robotparser import RobotFileParser  # standard Python tool that reads robots.txt

import requests
from bs4 import BeautifulSoup
import pdfplumber

In [ ]:
# Settings (change these here, not further down) 
ARCHIVE_URL = "https://www.priminfo.admin.ch/de/downloads/archiv"
# Only these premium years are downloaded. Our income data (ESTV) ends with 2022,
# so later premium years would have no income to compare with.
FIRST_YEAR = 2012
LAST_YEAR = 2022
# Pause between requests in seconds. A random value in this range is used each time,
# so we do not hit the server in a fixed rhythm.
MIN_WAIT = 2
MAX_WAIT = 5

# The notebook lives in notebooks/, but the data folder is one level up in the project.
# Path.cwd() is the folder the notebook is running in.
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PDF_DIR = RAW_DIR / "priminfo_pdfs"
PAGE_CACHE = RAW_DIR / "priminfo_archiv.html"
LOG_FILE = RAW_DIR / "download_log.csv"

# Create the folders if they do not exist yet (parents=True also creates data/ and data/raw/).
PDF_DIR.mkdir(parents=True, exist_ok=True)
print("[setup] project root:", PROJECT_ROOT)
print("[setup] PDFs will be saved in:", PDF_DIR)

[setup] project root: c:\Users\bitsc\projects\CIP-project
[setup] PDFs will be saved in: c:\Users\bitsc\projects\CIP-project\data\raw\priminfo_pdfs


### Helper functions

Two small functions that we use in every download

- **`log_request`** writes one line to `download_log.csv`: time, URL, status code
- **`polite_get`** downloads a URL with our User-Agent, logs it and then waits 2 to 5 seconds

Because every download goes through `polite_get`, we can't forget the pause or the log

> **HTTP status codes in short:** `200` = OK · `404` = page not found · `403` = access denied ·
> `500` = error on the server.


In [ ]:
# --- Helper functions ----------------------------------------------------

def log_request(url, status):
    """Write one line (time, URL, HTTP status) to data/raw/download_log.csv.

    The course asks us to document our requests. If the log file is new,
    a header line is written first.
    """
    is_new = not LOG_FILE.exists()
    with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if is_new:
            writer.writerow(["timestamp", "url", "status"])
        writer.writerow([datetime.now().isoformat(timespec="seconds"), url, status])


def polite_get(url):
    """Download one URL, log it, then wait 2-5 seconds.

    Returns the requests "response" object. raise_for_status() stops the
    notebook with an error if the server answers with e.g. 404 or 500,
    so we never silently save an error page as if it were a PDF.
    """
    response = requests.get(url, timeout=30)
    log_request(url, response.status_code)
    print(f"[download] {response.status_code}  {url}")
    time.sleep(random.uniform(MIN_WAIT, MAX_WAIT))
    response.raise_for_status()
    return response

## Step 1: Check robots.txt


In [29]:
host = urlparse(ARCHIVE_URL)            # splits the address into parts (scheme, host, path ...)
robots_url = f"{host.scheme}://{host.netloc}/robots.txt"

robots = RobotFileParser()
robots.set_url(robots_url)
robots.read()                           # downloads and reads robots.txt (a missing file counts as "allow all")

allowed = robots.can_fetch(USER_AGENT, ARCHIVE_URL)
print("[robots]", robots_url)
print("[robots] are we allowed to fetch the archive page?", allowed)

if not allowed:
    raise SystemExit("robots.txt forbids this page, so we stop here.")

[robots] https://www.priminfo.admin.ch/robots.txt
[robots] are we allowed to fetch the archive page? True


## Step 2: Download the archive page

### Why save a copy?
**Caching:** from the second run on, the notebook reads the saved file and doesn't ask the server
again. That is faster, kinder to the server, and the course requires it.
If you want a fresh copy, delete `data/raw/priminfo_archiv.html` and run the cell again.


In [30]:
if PAGE_CACHE.exists():
    print("[page] using cached copy:", PAGE_CACHE)
    html = PAGE_CACHE.read_text(encoding="utf-8")
else:
    html = polite_get(ARCHIVE_URL).text
    PAGE_CACHE.write_text(html, encoding="utf-8")
    print("[page] saved to:", PAGE_CACHE)

print("[page] length of the page in characters:", len(html))

[page] using cached copy: c:\Users\bitsc\projects\CIP-project\data\raw\priminfo_archiv.html
[page] length of the page in characters: 74603


## Step 3: Find all PDF links

### What happens
BeautifulSoup goes through the page and looks at every link (`<a href="...">`). We keep the links
whose address ends in `.pdf`.

### Good to know
- Links are often *relative*, e.g. `/dam/de/praemien_2012.pdf`. `urljoin` turns them into full
  addresses beginning with `https://...`.
- The same PDF can appear twice on a page -> drop the duplicates.

### What you should see
A count, then one line per PDF:
```
[links] found 24 PDF links, 12 unique
    Prämien 2012 -> https://www.priminfo.admin.ch/....pdf
```
(These numbers are only an example. The real numbers depend on the page.)

In [31]:
import re   # "regular expressions": a way to search text for patterns, here a 4-digit year

soup = BeautifulSoup(html, "html.parser")   # turns the HTML text into something we can search

pdf_links = []
for a in soup.find_all("a", href=True):     # every <a> tag that has an href attribute
    full_url = urljoin(ARCHIVE_URL, a["href"])
        # We only want the main premium overviews, named like "praemien_2012.pdf".
    # This skips "praemien_eu_..." (people living in the EU) and
    # "praemienverbilligung_..." (premium subsidies), which are not part of our research questions.
    file_name = Path(urlparse(full_url).path).name.lower()
    if not re.fullmatch(r"praemien_\d{4}\.pdf", file_name):   # fullmatch = the whole name must fit the pattern
        continue

    link_text = a.get_text(strip=True)      # the visible text of the link, e.g. "Prämien 2012"

    # Look for a year like 2012 first in the link text, then in the address.
    # \b means "word boundary", so 2012 is found but not the "2012" inside "120123".
    match = re.search(r"\b(20\d{2})\b", link_text) or re.search(r"(20\d{2})", full_url)
    if match is None:
        print("[links] no year found, skipped:", link_text, "->", full_url)
        continue

    year = int(match.group(1))              # group(1) is the year that was found, as a number
    if FIRST_YEAR <= year <= LAST_YEAR:
        pdf_links.append({"year": year, "url": full_url, "text": link_text})

# The same file can appear twice on a page. dict.fromkeys keeps the order and drops duplicates.
unique_urls = list(dict.fromkeys(link["url"] for link in pdf_links))

print(f"[links] {len(pdf_links)} PDF links between {2012} and {2022}, {len(unique_urls)} unique")
for link in sorted(pdf_links, key=lambda l: l["year"]):   # sorted by year, so the list is easy to check
    print("   ", link["year"], "|", link["text"], "->", link["url"])

[links] no year found, skipped: Prämienübersicht 1999PDF-Dokument -> https://www.priminfo.admin.ch/downloads/praemien_1999.pdf
[links] no year found, skipped: Prämienübersicht 1998PDF-Dokument -> https://www.priminfo.admin.ch/downloads/praemien_1998.pdf
[links] no year found, skipped: Prämienübersicht 1997PDF-Dokument -> https://www.priminfo.admin.ch/downloads/praemien_1997.pdf
[links] no year found, skipped: Prämienübersicht 1996PDF-Dokument -> https://www.priminfo.admin.ch/downloads/praemien_1996.pdf
[links] 11 PDF links between 2012 and 2022, 11 unique
    2012 | Prämienübersicht 2012PDF-Dokument -> https://www.priminfo.admin.ch/downloads/praemien_2012.pdf
    2013 | Prämienübersicht 2013PDF-Dokument -> https://www.priminfo.admin.ch/downloads/praemien_2013.pdf
    2014 | Prämienübersicht 2014PDF-Dokument -> https://www.priminfo.admin.ch/downloads/praemien_2014.pdf
    2015 | Prämienübersicht 2015PDF-Dokument -> https://www.priminfo.admin.ch/downloads/praemien_2015.pdf
    2016 | Prä

## Step 4: Download the PDFs

Every PDF from Step 3 is saved in `data/raw/priminfo_pdfs/`. The file name is taken from the
address.

### Why it is safe to run several times
Before downloading, we check whether the file already exists. If it does, it is skipped
(`[skip]`). So we never download the same file twice.


In [32]:
for url in unique_urls:
    file_name = Path(urlparse(url).path).name   # the last part of the address, e.g. "praemien_2012.pdf"
    target = PDF_DIR / file_name

    if target.exists():
        print("[skip] already downloaded:", file_name)
        continue

    response = polite_get(url)
    target.write_bytes(response.content)        # .content is the raw file, write_bytes saves it unchanged
    print("[saved]", target)

print("[done] PDFs in folder:", len(list(PDF_DIR.glob("*.pdf"))))

[skip] already downloaded: praemien_2022.pdf
[skip] already downloaded: praemien_2021.pdf
[skip] already downloaded: praemien_2020.pdf
[skip] already downloaded: praemien_2019.pdf
[skip] already downloaded: praemien_2018.pdf
[skip] already downloaded: praemien_2017.pdf
[skip] already downloaded: praemien_2016.pdf
[skip] already downloaded: praemien_2015.pdf
[skip] already downloaded: praemien_2014.pdf
[skip] already downloaded: praemien_2013.pdf
[skip] already downloaded: praemien_2012.pdf
[done] PDFs in folder: 11


---
## Step 5: Look inside one PDF

### What happens
We open **one** PDF with pdfplumber and print:
1. the number of pages,
2. the plain text of page 1,
3. the tables pdfplumber finds on page 1 (the first 8 rows of each).

### Why don't parse everything yet
Every PDF is built differently. The parsing code depends completely on the table layout: which
columns there are, where the canton is, how the deductibles are listed. So we **look first** and
write the parsing code afterwards.

### How to look at another file
Change `pdf_files[0]` to `pdf_files[1]`, `pdf_files[2]` and so on.

In [33]:
pdf_files = sorted(PDF_DIR.glob("*.pdf"))
print("[pdf] files available:", [p.name for p in pdf_files])

# Take the first file. Change the 0 to look at another one.
example = pdf_files[0]
print("[pdf] opening:", example.name)

with pdfplumber.open(example) as pdf:            # "with" closes the file again automatically
    print("[pdf] number of pages:", len(pdf.pages))
    first_page = pdf.pages[0]

    print("\n----- plain text of page 1 (first 1500 characters) -----")
    print((first_page.extract_text() or "")[:1500])

    print("\n----- tables pdfplumber detects on page 1 -----")
    tables = first_page.extract_tables()          # a list of tables, each table is a list of rows
    print("number of tables:", len(tables))
    for i, table in enumerate(tables):
        print(f"\ntable {i}: {len(table)} rows")
        for row in table[:8]:                     # only the first 8 rows, to keep it readable
            print("   ", row)

[pdf] files available: ['praemien_2012.pdf', 'praemien_2013.pdf', 'praemien_2014.pdf', 'praemien_2015.pdf', 'praemien_2016.pdf', 'praemien_2017.pdf', 'praemien_2018.pdf', 'praemien_2019.pdf', 'praemien_2020.pdf', 'praemien_2021.pdf', 'praemien_2022.pdf']
[pdf] opening: praemien_2012.pdf
[pdf] number of pages: 1191

----- plain text of page 1 (first 1500 characters) -----
Eidgenössisches Departement des Innern EDI
Département fédéral de l'intérieur DFI CH
Bundesamt für Gesundheit BAG
Office fédéral de la santé publique OFSP
Oktober/octobre/ottobre 2011
Prämienübersicht
Aperçu des primes
Panoramica dei premi
2012
- Grundversicherungsprämien (Wahlfranchisen, HMO, Hausarzt und
weitere Modelle)
- Kündigungsfristen
- Sparmöglichkeiten
- Primes de l’assurance de base (franchises à option, HMO, médecin
de famille et autres modèles)
- Délais de résiliation
- Possibilités d’économies
- Premi dell’assicurazione di base (franchigie opzionale, HMO, medico
di famiglia e altri modelli)
- Termini di d

## Step 6: Find the pages with the premium tables

### What happens
We look at a few pages spread through `praemien_2012.pdf`. For each page we print the text and the
first rows of any table pdfplumber detects.

### Why
The PDF has over 1000 pages, and page 1 is only the cover (no tables). Before we can write the
parsing code, we need to know **where** the premium tables start and **how** they are laid out:
which columns there are, and where the insurer, canton, region, age group and deductible are.

### Good to know
pdfplumber counts pages from **0**, so page 10 of the PDF is `pdf.pages[9]`. The code takes care of
this, so you can write normal page numbers in `PAGES_TO_LOOK_AT`.

In [34]:
# Look at a few pages further inside the PDF, to find where the premium tables start.
# Page numbers in pdfplumber start at 0, so pdf.pages[9] is page 10 of the PDF.
PAGES_TO_LOOK_AT = [2, 5, 10, 20, 50, 100]

example = PDF_DIR / "praemien_2012.pdf"

with pdfplumber.open(example) as pdf:
    for page_number in PAGES_TO_LOOK_AT:
        page = pdf.pages[page_number - 1]   # minus 1, because counting starts at 0
        text = page.extract_text() or ""
        tables = page.extract_tables()

        print("=" * 80)
        print(f"PAGE {page_number}   |   tables found: {len(tables)}")
        print("=" * 80)
        print(text[:800])                   # first 800 characters, so the output stays short

        if tables:
            print("\n--- first rows of the first table ---")
            for row in tables[0][:6]:
                print("   ", row)
        print()

PAGE 2   |   tables found: 0


PAGE 5   |   tables found: 0
Krankenversicherung: Information für die Versicherten
Das Krankenversicherungsgesetz (KVG) bietet Ihnen verschiedene Möglichkeiten, Ihre persönliche
Versicherungsprämie zu optimieren. Die vorliegende Information, vor allem das Kapitel „Sparmöglich-
keiten“, soll Ihnen dabei helfen und Ihnen zeigen, wie Sie Ihre Prämie spürbar senken können, ohne
Qualitätseinbussen beim Versicherungsschutz in Kauf nehmen zu müssen.
Auf dem Internet finden Sie zudem einen elektronischen Prämienvergleich, mit dem Sie die Prämien in
der Grundversicherung mit den verschiedenen Franchisen-Stufen der Krankenkassen in Ihrer Region
vergleichen können. Auch die Prämien bei HMO-, Hausarzt- (HAM) sowie weiteren Versicherungsmo-
dellen sind darin ersichtlich.
Die Internet-Adresse lautet: www.priminfo.ch.
Wünschen Sie w

PAGE 10   |   tables found: 0
Termini di disdetta
Cambiamento della cassa malati
- Avete la possibilità di stipulare l’assicurazione di ba

## Step 7: Read all premium rows out of one PDF

### What happens
We go through **every page** of `praemien_2012.pdf` and read the premium rows line by line.
For each page we remember the model, the canton, the age group and the region from the page
header. Then each data row becomes 12 rows in our table: 6 deductibles with accident and
6 deductibles without accident.

### Why we read the text and not the tables
pdfplumber found no tables (Step 6), because the PDF has no lines around its cells. The text,
however, is very regular, so we can read it with *regular expressions*: patterns that describe
what a line looks like, e.g. "6 numbers, then text, then 6 numbers".

### Units
The premiums in the PDF are **CHF per month**.

In [38]:
import re
import pdfplumber
import pandas as pd

# Patterns: what the lines in the PDF look like 
NUMBER = re.compile(r"^\d+\.\d$")       # one premium, e.g. "180.2"
DEDUCTIBLE = re.compile(r"^\d+$")       # one deductible in the header, e.g. "300"
CANTON_PATTERN = re.compile(r"^(20\d{2}) ([A-Z]{2})$")           # e.g. "2012 AI"
AGE_PATTERN = re.compile(r"Franchise / Franchise:\s*(.+?)\s*/")   # e.g. "Junge Erwachsene"

LINE_TOLERANCE = 3    # words whose vertical position differs by less than this (in points) are on the same line


def group_into_lines(words):
    """Put the words of a page together into lines, from top to bottom.

    pdfplumber gives every word with its position on the page:
    x0/x1 = left/right edge, top = distance from the top of the page.
    Words with (almost) the same `top` belong to the same line.
    """
    lines = []
    for word in sorted(words, key=lambda w: (round(w["top"]), w["x0"])):
        if lines and abs(lines[-1][0]["top"] - word["top"]) < LINE_TOLERANCE:
            lines[-1].append(word)
        else:
            lines.append([word])
    return [sorted(line, key=lambda w: w["x0"]) for line in lines]   # each line from left to right


def middle(word):
    """Horizontal middle of a word on the page (in points)."""
    return (word["x0"] + word["x1"]) / 2


def nearest_column(word, columns):
    """Return the deductible whose header is closest to this premium, and the distance.

    `columns` is a list of (deductible, x position of the header).
    This is how we know that a premium belongs to e.g. CHF 2500 even when
    the insurer does not offer CHF 2000 and that column is empty.
    """
    deductible, x = min(columns, key=lambda col: abs(col[1] - middle(word)))
    return deductible, abs(x - middle(word))


def parse_page(page, page_number, state, rows, problems):
    """Read all premium rows on one PDF page and add them to `rows`.

    `state` remembers the header information (model, canton, age group, region,
    deductible positions). We keep it from page to page, because a table can
    continue on the next page without a new header.
    Rows we cannot read safely go to `problems`, so we see what we lose.
    """
    for line in group_into_lines(page.extract_words()):
        texts = [w["text"] for w in line]
        text = " ".join(texts)

        # Page title, e.g. "Prämien - HMO". "Prämien," and "Prämienübersicht" are explanation pages.
        if text.startswith("Prämien") and not text.startswith(("Prämien,", "Prämienübersicht")):
            state["model"] = text.replace("Prämien", "", 1).strip(" -") or "(no model name)"
            continue

        match = CANTON_PATTERN.match(text)
        if match:
            state["year"] = int(match.group(1))
            state["canton"] = match.group(2)
            continue

        if text.startswith("Franchise / Franchise:"):
            match = AGE_PATTERN.search(text)
            if match:
                state["age_group"] = match.group(1)
            continue

        # Header with the deductibles: "300 500 ... Region / Région 0 300 500 ..."
        # Only if a region NUMBER follows "Région". Other tables in the PDF also contain the
        # word Région and those are not our header
        if "Région" in texts and texts.index("Région") + 1 < len(texts) and texts[texts.index("Région") + 1].isdigit():
            pos = texts.index("Région")
            left_words, right_words = line[:pos], line[pos + 2:]   # pos+1 is the region number
            state["region"] = int(texts[pos + 1])
            # Remember each deductible AND where it stands on the page, so we can match premiums to it.
            state["cols_mit"] = [(int(w["text"]), middle(w)) for w in left_words if DEDUCTIBLE.match(w["text"])]
            state["cols_ohne"] = [(int(w["text"]), middle(w)) for w in right_words if DEDUCTIBLE.match(w["text"])]
            # A premium may be at most half a column away from its header, otherwise we don't trust the match.
            state["max_distance"] = min(b[1] - a[1] for a, b in zip(state["cols_mit"], state["cols_mit"][1:])) / 2
            continue

        # A data row starts and ends with a premium; everything in between is the insurer.
        if len(line) < 3 or not NUMBER.match(texts[0]) or not NUMBER.match(texts[-1]):
            continue
        n_left = 0
        while NUMBER.match(texts[n_left]):
            n_left += 1
        n_right = 0
        while NUMBER.match(texts[-1 - n_right]):
            n_right += 1
        insurer = " ".join(texts[n_left:len(texts) - n_right])

        if "cols_mit" not in state:
            problems.append((page_number, "row before any deductible header", text))
            continue

        new_rows = []
        ok = True
        for accident, words, columns in [("mit_unfall", line[:n_left], state["cols_mit"]),
                                         ("ohne_unfall", line[len(line) - n_right:], state["cols_ohne"])]:
            used = set()
            for word in words:
                deductible, distance = nearest_column(word, columns)
                # Too far from every header, or two premiums for the same deductible: we do not trust it.
                if distance > state["max_distance"] or deductible in used:
                    ok = False
                used.add(deductible)
                new_rows.append({
                    "year": state.get("year"), "canton": state.get("canton"),
                    "region": state.get("region"), "age_group": state.get("age_group"),
                    "model": state.get("model"), "insurer": insurer,
                    "accident": accident, "deductible": deductible,
                    "premium_month": float(word["text"]), "page": page_number,
                })
        if ok:
            rows.extend(new_rows)
        else:
            problems.append((page_number, "premium could not be matched to a deductible", text))


# Run it on the whole 2012 PDF 
example = PDF_DIR / "praemien_2012.pdf"
rows, problems, state = [], [], {}

with pdfplumber.open(example) as pdf:
    for i, page in enumerate(pdf.pages, start=1):      # enumerate counts along: i = 1, 2, 3, ...
        parse_page(page, i, state, rows, problems)
        if i % 100 == 0:
            print(f"[parse] {i} of {len(pdf.pages)} pages done")

premiums = pd.DataFrame(rows)    # turn the list of rows into a table

print("\n[result] rows read:", len(premiums))
print("[result] problem lines:", len(problems))
print("\n[result] models:\n", premiums["model"].value_counts().to_string())
print("\n[result] age groups:\n", premiums["age_group"].value_counts().to_string())
print("\n[result] cantons found:", premiums["canton"].nunique(), sorted(premiums["canton"].dropna().unique()))

# On which pages do the "Krankenpflege - ..." models appear? We look at them before deciding what they are.
kp = premiums[premiums["model"].str.startswith("Krankenpflege", na=False)]
print("\n[check] 'Krankenpflege' models on pages", kp["page"].min(), "to", kp["page"].max())

print("\n[result] first 10 problem lines:")
for page_number, reason, line in problems[:10]:
    print(f"   page {page_number}: {reason}\n      {line}")

[parse] 100 of 1191 pages done
[parse] 200 of 1191 pages done
[parse] 300 of 1191 pages done
[parse] 400 of 1191 pages done
[parse] 500 of 1191 pages done
[parse] 600 of 1191 pages done
[parse] 700 of 1191 pages done
[parse] 800 of 1191 pages done
[parse] 900 of 1191 pages done
[parse] 1000 of 1191 pages done
[parse] 1100 of 1191 pages done

[result] rows read: 260051
[result] problem lines: 578

[result] models:
 model
Hausarztmodell                                  107530
Grundversicherung                                67407
Weitere Versicherungsmodelle                     56442
HMO                                              22196
Krankenpflege - Hausarztmodell                    2360
Krankenpflege - Grundversicherung                 2148
Krankenpflege - Weitere Versicherungsmodelle      1968

[result] age groups:
 age_group
Kinder              95811
Erwachsene          82120
Junge Erwachsene    82120

[result] cantons found: 26 ['AG', 'AI', 'AR', 'BE', 'BL', 'BS', 'FR', 'GE', 'GL

## Step 8: Read all PDFs (2012–2022) and save the result

We run the same `parse_page` function from Step 7 on every premium PDF, `praemien_2012.pdf`
to `praemien_2022.pdf`, and combine everything into one table. The table is saved as a CSV in
`data/processed/`.

### Why Step 7 used only 2012
Testing on one PDF first is much faster. Now that the code works for 2012, we use it for all years.

### Check the problem lines per year
If one year has far more problem lines than the others, the PDF layout probably changed in that
year. Then we have to adjust the code for that year.

In [ ]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = PROCESSED_DIR / "priminfo_premiums_2012_2022.csv"

# Only the main premium PDFs, e.g. "praemien_2012.pdf". The glob pattern
# "praemien_20??.pdf" means: "praemien_20", then exactly 2 characters, then ".pdf".
# That skips "praemien_eu_..." and "praemienverbilligung_...".
pdf_files = sorted(PDF_DIR.glob("praemien_20??.pdf"))
# QUICK TEST: only the first and the last year. Delete this line to read all PDFs.
pdf_files = [pdf_files[0], pdf_files[-1]]
print("[files]", [p.name for p in pdf_files])

all_tables = []       # one table per PDF, combined at the end
summary = []          # rows read and problems per year, for checking

for pdf_path in pdf_files:
    print(f"\n[parse] {pdf_path.name}")
    # Start with an empty state for every PDF, so nothing from the previous year carries over.
    rows, problems, state = [], [], {}

    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            parse_page(page, i, state, rows, problems)
            if i % 200 == 0:
                print(f"   {i} of {len(pdf.pages)} pages done")

    table = pd.DataFrame(rows)
    table["source_file"] = pdf_path.name     # so we always know which PDF a row comes from
    all_tables.append(table)

    summary.append({
        "file": pdf_path.name,
        "rows_read": len(table),
        "problem_lines": len(problems),
        "cantons": table["canton"].nunique() if len(table) else 0,
    })
    print(f"   rows read: {len(table)}, problem lines: {len(problems)}")

# pd.concat glues the tables of all years together, one below the other.
premiums_all = pd.concat(all_tables, ignore_index=True)
premiums_all.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

print("\n[saved]", OUTPUT_FILE, "with", len(premiums_all), "rows")
print("\n[summary] per PDF:")
print(pd.DataFrame(summary).to_string(index=False))

[files] ['praemien_2012.pdf', 'praemien_2022.pdf']

[parse] praemien_2012.pdf
   200 of 1191 pages done
   400 of 1191 pages done
   600 of 1191 pages done
   800 of 1191 pages done
   1000 of 1191 pages done
   rows read: 260051, problem lines: 578

[parse] praemien_2022.pdf
   200 of 1254 pages done
   400 of 1254 pages done
   600 of 1254 pages done
